# Notebook 00 â€” Setup & Orientation
## Sarvam AI Ã— Indic NLP: A *Speech and Language Processing* Companion

**Purpose:** Verify your environment, meet the Sarvam model family, and see why Indic NLP is fundamentally different from the English-centric world of the textbook.

---
**Prerequisites:** Run `pip install -r ../requirements.txt` and create a `.env` file in the repo root:
```
SARVAM_API_KEY=<your_key_here>
DEMO_MODE=True
```

### Install/import check, load .env, init client

This cell verifies your Python environment, loads API credentials from `.env`, and initialises the Sarvam AI client. If anything fails here, fix it before proceeding — every subsequent cell depends on this setup.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from utils.sarvam_helpers import load_client, reset_demo_counters
from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES, ENGLISH_TRANSLATIONS
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

reset_demo_counters()
client = load_client()
print('âœ“ Environment OK â€” SarvamAI client ready')
print(f'âœ“ DEMO_MODE = {os.getenv("DEMO_MODE")}')

## Why Indic NLP is Hard â€” Background & Framing

Jurafsky & Martin (*Speech and Language Processing* opens with the ambition: *"Every question you ever wanted to ask about language."* But the textbook's running examples are almost entirely **English**. When we shift to the Indian subcontinent, every assumption breaks:

| Challenge | English assumption | Indic reality |
|-----------|-------------------|---------------|
| **Scripts** | 1 script (Latin) | 10+ scripts (Devanagari, Tamil, Telugu, Bengali, Gujaratiâ€¦) |
| **Morphology** | Relatively isolating | Agglutinative (Tamil, Telugu) or fusional (Hindi, Sanskrit) |
| **Word order** | SVO fixed | SOV default; Hindi allows scrambling |
| **Tokenization** | Whitespace works | Sandhi (Hindi), compound words (Tamil) break whitespace |
| **Code-mixing** | Rare | 10M+ speakers mix Hindi+English per sentence |
| **Resources** | 100B+ token corpora | Many languages: < 1B tokens |

**India has 22 scheduled languages** under the Constitution, written in at least 10 scripts. The top 8 languages by speaker count (Hindi, Bengali, Telugu, Tamil, Gujarati, Urdu, Kannada, Malayalam) together have **~1.2 billion speakers** â€” but most NLP research ignores them.

**Sarvam AI** was founded to fix this. Their model suite (Sarvam-M 24B, Bulbul TTS, Saaras STT, Mayura MT) is trained specifically on Indic language data from AI4Bharat corpora and proprietary sources.

### Language detection smoke test on all 4 sample texts

A **smoke test** is a quick sanity check that the API is responding correctly. We send four sample texts (Hindi, Tamil, Bengali, Telugu) to the Language Detection endpoint and verify it identifies each one.


<details>
<summary><strong>What is Language Identification (LID)?</strong></summary>

Language Identification is the task of determining which natural language a piece of text is written in. It's typically the **first step in any multilingual NLP pipeline** — you need to know the language before you can tokenize, translate, or parse.

**How it works under the hood:**
- **Character n-gram models**: Count frequencies of character sequences (e.g., 'tion' is common in English, 'keit' in German). Works well for scripts with unique character sets.
- **Unicode block detection**: For Indic languages, simply checking which Unicode range characters fall in gives near-perfect accuracy (Tamil characters only appear in U+0B80–0BFF).
- **Statistical classifiers**: For closely related languages sharing a script (Hindi vs Marathi, both Devanagari), models use word-level features and n-gram distributions.

**Sarvam's LID** supports all 22 scheduled Indian languages and returns a confidence score.
</details>


In [ ]:
reset_demo_counters()
from utils.sarvam_helpers import detect_language

results = []
for lang_code, text in SAMPLE_TEXTS.items():
    try:
        det = detect_language(client, text)
        results.append({
            'Expected': LANGUAGE_NAMES[lang_code],
            'Expected Code': lang_code,
            'Detected': det['language_code'],
            'Confidence': f"{det['confidence']:.2%}",
            'Text Preview': text[:40] + 'â€¦'
        })
    except Exception as e:
        results.append({'Expected': lang_code, 'Error': str(e)})

df = pd.DataFrame(results)
print('Language Detection Results:')
print(df.to_string(index=False))

### Unicode codepoint range bar chart (script diversity visualization)

This chart visualizes **why Indic NLP is fundamentally different from English NLP** at the encoding level.

Each bar represents a script's Unicode block — its starting codepoint and range width. Latin (English) occupies a tiny, low range, while India's 10+ scripts are spread across completely separate Unicode blocks.

**Why this matters:**

1. **India uses 10+ separate Unicode blocks** — unlike European languages that mostly share Latin. A single character-level model can't just learn one alphabet; it needs to handle widely scattered codepoint ranges.

2. **Tokenizers built for Latin break on Indic text** — BPE/WordPiece models trained on English see Indic characters as rare byte sequences, leading to over-fragmentation (we explore this in depth in notebook 01).

3. **Script detection is trivial via codepoint ranges** — you can identify which language a text is written in just by checking which Unicode block its characters fall into. This is essentially what the Language Detection API does under the hood.

4. **This diversity is the theme of the entire course** — India's linguistic richness isn't just about vocabulary differences; it's about fundamentally different writing systems, grammars, and morphologies. Every notebook that follows will show you where English-centric NLP assumptions break down in instructive ways.


In [ ]:
reset_demo_counters()

# Unicode block ranges — Indic scripts + global scripts for comparison
script_ranges = {
    # --- Global scripts (for comparison) ---
    'Latin (English)':    (0x0041, 0x007A),   # A-z
    'Arabic':             (0x0600, 0x06FF),
    'CJK (Chinese)':      (0x4E00, 0x9FFF),   # CJK Unified Ideographs
    'Hangul (Korean)':    (0xAC00, 0xD7AF),   # Hangul Syllables
    'Cyrillic (Russian)': (0x0400, 0x04FF),
    'Thai':               (0x0E00, 0x0E7F),
    # --- Indic scripts ---
    'Devanagari (Hindi)': (0x0900, 0x097F),
    'Bengali':            (0x0980, 0x09FF),
    'Gurmukhi (Punjabi)': (0x0A00, 0x0A7F),
    'Gujarati':           (0x0A80, 0x0AFF),
    'Odia':               (0x0B00, 0x0B7F),
    'Tamil':              (0x0B80, 0x0BFF),
    'Telugu':             (0x0C00, 0x0C7F),
    'Kannada':            (0x0C80, 0x0CFF),
    'Malayalam':          (0x0D00, 0x0D7F),
}

labels = list(script_ranges.keys())
starts = [v[0] for v in script_ranges.values()]
sizes  = [v[1] - v[0] for v in script_ranges.values()]

# Color: blue shades for global, warm palette for Indic
n_global = 6
n_indic = len(labels) - n_global
colors_global = plt.cm.Blues([0.3, 0.35, 0.4, 0.45, 0.5, 0.55])[:n_global]
colors_indic  = plt.cm.Set2.colors[:n_indic]
colors = list(colors_global) + list(colors_indic)

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(labels, sizes, left=starts, color=colors, edgecolor="white", height=0.7)

# Add a visual separator between global and Indic
ax.axhline(y=n_global - 0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(0x0020, n_global - 0.5, "  ── Indic scripts below ──", va="center",
        fontsize=9, color="gray", style="italic")

ax.set_xlabel("Unicode Codepoint")
ax.set_title("Unicode Block Ranges: Indic Scripts vs Global Scripts
"
             "(bar width = number of codepoints in block)", fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"U+{int(x):04X}"))
import seaborn as sns
sns.despine()
plt.tight_layout()
plt.savefig("../outputs/figures/00_unicode_ranges.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved to outputs/figures/00_unicode_ranges.png")


## Sarvam Model Family Overview

| Model | Type | Key Capability | API Endpoint |
|-------|------|---------------|--------------|
| **Sarvam-M 24B** | LLM (reasoning) | 22 Indic languages, math/code, structured output | `chat.completions` |
| **Mayura v1** | Neural MT | High-quality translation, 4 style modes | `text.translate` |
| **Sarvam-Translate v1** | Neural MT | Alternative MT model for comparison | `text.translate` |
| **Bulbul v3** | TTS | 11 Indian languages, temperature-based expressiveness | `text_to_speech.convert` |
| **Bulbul v2** | TTS (legacy) | Pitch/loudness control (older API style) | `text_to_speech.convert` |
| **Saaras v3** | ASR | ~19% WER, 4 transcription modes | `speech_to_text.transcribe` |
| **Sarvam Vision 3B** | Multimodal | Document intelligence, structured extraction | `vision` |

### Model Selection Guide
- **Translation task** â†’ Mayura v1 (formal/colloquial modes) or Sarvam-Translate v1
- **Text generation / reasoning in Hindi** â†’ Sarvam-M 24B (`reasoning_effort=low` for speed)
- **Voice applications** â†’ Bulbul v3 (temperature control) 
- **Transcription** â†’ Saaras v3 (verbatim, transliterate, or code-mixed modes)
- **Document OCR + extraction** â†’ Sarvam Vision 3B

### Textbook Connection
Each model maps to a chapter in the textbook:
- Mayura / Sarvam-Translate â†’ **Ch 13** (Machine Translation)
- Sarvam-M â†’ **Ch 10** (Transformers), **Ch 11** (Fine-tuning)
- Bulbul â†’ **Ch 16** (TTS / Speech Synthesis)
- Saaras â†’ **Ch 16** (ASR / Automatic Speech Recognition)

### Display sample texts with their English translations

These four texts are the **canonical examples** used throughout all 7 notebooks. They were chosen to represent four different language families, scripts, and morphological typologies:

| Language | Script | Family | Morphology |
|----------|--------|--------|------------|
| Hindi | Devanagari | Indo-Aryan | Moderately inflectional |
| Tamil | Tamil | Dravidian | Agglutinative |
| Bengali | Bengali | Indo-Aryan | Inflectional |
| Telugu | Telugu | Dravidian | Agglutinative |

Each text says roughly the same thing in the respective language, allowing direct cross-linguistic comparisons in later notebooks.


In [ ]:
reset_demo_counters()

print('='*70)
print('CANONICAL SAMPLE TEXTS â€” used throughout all notebooks')
print('='*70)
for lang_code, text in SAMPLE_TEXTS.items():
    lang_name = LANGUAGE_NAMES[lang_code]
    translation = ENGLISH_TRANSLATIONS[lang_code]
    print(f'\n[{lang_name} / {lang_code}]')
    print(f'  {text}')
    print(f'  â†’ (EN) {translation}')
print('\nâœ“ Setup complete! Proceed to notebook 01_tokenization_morphology.ipynb')

---
## Google Gemini — A Global Multilingual Baseline

> **Note:** Gemini comparison cells require `GEMINI_API_KEY` set in `.env`.
> If absent, cells skip gracefully with a warning.
>
> Get a free key at: https://aistudio.google.com/apikey

**Why compare Sarvam with Gemini?**

Google's **Gemini 2.0 Flash** is a frontier multilingual model trained on 100+ languages — including Hindi, Tamil, Bengali, and Telugu. It's fast, capable, and free-tier friendly. But it's a *generalist*: it wasn't built specifically for Indic languages.

Sarvam, by contrast, is an **Indic specialist** — trained on AI4Bharat corpora with deep understanding of code-mixing, honorific registers, and script-specific phenomena.

**This comparison is instructive because it reveals where specialisation matters:**

| Dimension | Gemini 2.0 Flash | Sarvam-M 24B |
|-----------|------------------|--------------|
| **Languages** | 100+ (global) | 22 Indic (deep) |
| **Hindi idioms** | Often translates literally | Understands cultural context |
| **Honorific register** | May miss formal/informal distinction | Trained on register-aware data |
| **Code-mixing** | Handles common patterns | Native Hinglish understanding |
| **Speech pipeline** | Separate APIs | Integrated (Bulbul TTS, Saaras STT) |

The next cell tests 3 Hindi scenarios where Indic specialisation should shine.

In [ ]:
reset_demo_counters()

# ── Sarvam vs Gemini: Hindi showdown ──────────────────────────────────────
# Three test cases where Indic specialisation should outperform a global model.

from utils.gemini_helpers import load_gemini_client, gemini_chat
from utils.sarvam_helpers import chat_complete

gemini = load_gemini_client()
sarvam = client  # already loaded in cell 2

test_cases = [
    {
        "label": "1. Hindi Idiom — 'दाल में कुछ काला है'",
        "prompt": "Explain the Hindi idiom 'दाल में कुछ काला है' in English. "
                  "What does it really mean (not a literal translation)?",
        "why": "Gemini often translates literally ('there is something black in the lentils') "
               "instead of conveying 'something is fishy / suspicious'.",
    },
    {
        "label": "2. Honorific Register — formal 'आप' vs informal 'तुम'",
        "prompt": "Translate to English, preserving the politeness level:\n"
                  "'आप कल आइएगा, हम आपकी प्रतीक्षा करेंगे।'",
        "why": "Hindi has 3 register levels (तू/तुम/आप). Sarvam should preserve "
               "the formal register; Gemini may flatten it.",
    },
    {
        "label": "3. Code-Mixed Hinglish",
        "prompt": "Understand this Hinglish sentence and reply in pure Hindi:\n"
                  "'Mujhe kal tak report submit karni hai, deadline miss nahi kar sakta.'",
        "why": "Romanised Hindi mixed with English (common in Indian workplaces). "
               "Tests understanding of code-mixed input.",
    },
]

for tc in test_cases:
    print(f"\n{'='*70}")
    print(f"TEST: {tc['label']}")
    print(f"WHY:  {tc['why']}")
    print(f"{'='*70}")
    messages = [{"role": "user", "content": tc["prompt"]}]

    # Gemini first (baseline)
    if gemini is not None:
        try:
            g_resp = gemini_chat(gemini, messages, temperature=0.3)
            print(f"\n  🔵 Gemini 2.0 Flash (baseline):\n    {(g_resp or 'N/A')[:300]}")
        except Exception as e:
            print(f"\n  🔵 Gemini: [Error: {e}]")
    else:
        print("\n  🔵 Gemini: [Skipped — GEMINI_API_KEY not set]")

    # Sarvam second (Indic specialist)
    try:
        s_resp = chat_complete(sarvam, messages, reasoning_effort="low")
        if "<think>" in s_resp:
            s_resp = s_resp.split("</think>")[-1].strip()
        print(f"\n  🟠 Sarvam-M (Indic specialist):\n    {s_resp[:300]}")
    except Exception as e:
        print(f"\n  🟠 Sarvam-M: [Error: {e}]")

print(f"\n{'='*70}")
print("✓ Hindi showdown complete — compare outputs above.")